In [ ]:
import numpy as np
import rasterio
import xgboost as xgb

# 1. Paths to your 6 raster files (GeoTIFFs) for years 1-6
raster_files = [
    'year1.tif',
    'year2.tif',
    'year3.tif',
    'year4.tif',
    'year5.tif',
    'year6.tif'
]

# 2. Load rasters into a list of arrays
arrays = []
for f in raster_files:
    with rasterio.open(f) as src:
        arrays.append(src.read(1))  # Read first band

# 3. Stack arrays (rows=pixels, columns=years)
stacked = np.stack(arrays, axis=-1)  # shape: (rows, cols, 6)
rows, cols, years = stacked.shape
X_all = stacked.reshape(rows*cols, years)  # shape: (pixels, 6)

# 4. Recursive prediction: 
# Train on years 1-5 to predict year 6
X_train = X_all[:, :5]  # years 1-5
y_train = X_all[:, 5]   # year 6

model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    objective='reg:squarederror'
)
model.fit(X_train, y_train)

# Predict year 6 (can also compare with actual year 6 if desired)
pred_year6 = model.predict(X_train)

# Now predict year 7 using years 2-6 (replace year6 with predicted)
X_for_year7 = np.column_stack([X_all[:,1:5], pred_year6])
pred_year7 = model.predict(X_for_year7)

# 5. Save predicted year 7 as raster
with rasterio.open(raster_files[0]) as src:
    meta = src.meta.copy()
meta.update(dtype=rasterio.float32, count=1)

pred_raster = pred_year7.reshape(rows, cols)
with rasterio.open('predicted_year7.tif', 'w', **meta) as dst:
    dst.write(pred_raster.astype(rasterio.float32), 1)

print("Year 7 prediction saved as predicted_year7.tif")